In [1]:
from pathlib import Path

from src.minbpe import RegexTokenizer
#from src.gpt import GPTLanguageModel
from src.transformer.model_relative_positional_encoding import GPTLanguageModel

import torch

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_dir = Path("data") / "tokenizer"
checkpoint_dir = Path("data") / "ch09"

In [3]:
tokenizer = RegexTokenizer()
tokenizer.load(model_file=str(tokenizer_dir / "tokenizer.model"))

#ckpt_files = sorted(
#    checkpoint_dir.glob("checkpoint_*.pt"),
#    key=lambda x: x.stat().st_ctime,
#    #key=lambda x: int(x.name.split("-")[1]),
#    reverse=True,
#)
#checkpoint_path = ckpt_files[0]

checkpoint_path = checkpoint_dir / "fine-tuning_000068.pt"

print(f"load checkpoint: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path, weights_only=True, map_location=device)
parameters = checkpoint['meta']['parameters']

load checkpoint: data/ch09/fine-tuning_000068.pt


In [4]:
model = GPTLanguageModel(
    vocab_size=parameters['vocab_size'],
    block_size=parameters['block_size'],
    n_embd=parameters['n_embd'],
    n_head=parameters['n_head'],
    n_layer=parameters['n_layer'],
    dropout=parameters['dropout'],
    ignore_index=tokenizer.special_tokens["<|padding|>"],
    device=device,
)

model = torch.compile(model)
model.load_state_dict(checkpoint["model_state_dict"])

num_parameters = sum(p.numel() for p in model.parameters()) / 1e6
print(f'--> {num_parameters:_.3}M parameters')
# print_model_structure(model)

_ = model.eval()

--> 13.7M parameters


In [5]:
def into_tokens(role: str, content: str) -> torch.Tensor:
    d = tokenizer.special_tokens
    #print("~~~", d)

    input_msg = f"<|startoftext|>{role}<|separator|>{content}<|endoftext|>"
    print(f"--> {role} message: {input_msg}")

    input_tokens = tokenizer.encode(input_msg, allowed_special="all")

    return torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)


def ask_llm(input_tokens):
    model_answer = ""

    while True:
        output_tokens = model.generate(input_tokens=input_tokens, max_new_tokens=1)
        last_generated_token = output_tokens[0, -1].item()

        #print("~~~", input_tokens[0])
        model_answer += tokenizer.decode([last_generated_token])

        if last_generated_token == tokenizer.special_tokens["<|endoftext|>"]:
            break

        input_tokens = torch.cat((input_tokens, output_tokens[:, -1:]), dim=1)

        #if len(output_tokens[0]) > parameters['block_size']:
        #    break
        if len(input_tokens[0]) > parameters['block_size']:
            input_tokens = input_tokens[:, -parameters['block_size']:]

    return model_answer

def ask_llm_advanced(input_tokens):
    model_answer = ""

    while True:
        output_tokens = model.advanced_generation(
            input_tokens=input_tokens, max_new_tokens=1,
            temperature=.9, top_k=50, top_p=None,
        )

        last_generated_token = output_tokens[0, -1].item()

        #print("~~~", input_tokens[0])
        model_answer += tokenizer.decode([last_generated_token])

        if last_generated_token == tokenizer.special_tokens["<|endoftext|>"]:
            break

        input_tokens = torch.cat((input_tokens, output_tokens[:, -1:]), dim=1)

        #if len(output_tokens[0]) > parameters['block_size']:
        #    break
        if len(input_tokens[0]) > parameters['block_size']:
            input_tokens = input_tokens[:, -parameters['block_size']:]

    return model_answer

In [6]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

with torch.no_grad():
    output = model.generate(input_tokens=input_tokens, max_new_tokens=256)
    print("<-- assistant message:", tokenizer.decode(output[0].tolist()))

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|><|startoftext|>assistant<|separator|>What's a conlang?<|endoftext|><|startoftext|>user<|separator|>I want to be the 60s<|endoftext|><|startoftext|>assistant<|separator|>The Language Assistant program is a free and open source project that people could built upon new ideas and original software. It's useful to know what the model architecture was, and to what was behind it. But as an AI assistant, it can learn and improve my abilities.<|endoftext|><|startoftext|>assistant<|separator|>A goes like ChatGPT doesn't have any real-world examples, and the model is powerful and not yet exactly absolutely able to use ChatGPT to generate responses<|endoftext|><|startoftext|>assistant<|separator|>ChatGPT is like a chatbot.<|endoftext|><|startoftext|>assistant<|separator|>Do you want me to help you start writing?<|endoftext|><|startoftext|

In [7]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

answer = ask_llm(input_tokens)
print(f"<-- assistant message: {answer}")

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>assistant<|separator|>You use ChatGPT by OpenAI, a non-profit organization whose organization helps us make better predictions. Open Assistant, a large language model (LLM) created by LAION, is a professional assistant that can understand and respond to natural language input, help users make it answers on correct categories.<|endoftext|>


In [8]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

answer = ask_llm(input_tokens)
print(f"<-- assistant message: {answer}")

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>assistant<|separator|>Variational Generatures Problem are still in development and may not speculate on where in the future may come out on top of generation<|endoftext|>


In [9]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

answer = ask_llm_advanced(input_tokens)
print(f"<-- assistant message: {answer}")

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>assistant<|separator|>Your ChatGPT is an open source project which means that an open source software creates a community of developers, including the developers and navigating tools, as well as being an open source project. ChatGPT is a large language model created by LAION.<|endoftext|>


In [10]:
user_content = "什么是 ChatGPT?"
input_tokens = into_tokens("user", user_content)

answer = ask_llm(input_tokens)
print(f"<-- assistant message: {answer}")

--> user message: <|startoftext|>user<|separator|>什么是 ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>assistant<|separator|>I��m sorry, but I don��t know what to do when I��m not able to read like this.<|endoftext|>
